# IUL Crediting Methodology

This notebook explains how OPIE computes interest crediting for Indexed Universal Life (IUL) policies.

## How IUL Crediting Works

Unlike traditional UL (which applies a single crediting rate to the entire account value),
IUL splits the account value across multiple **index accounts**, each with its own crediting strategy.

```
  Account Value ($100,000)
  ┌─────────────────────────────────────────────────┐
  │  Fixed Account (40%)    │  S&P 500 PTP (60%)        │
  │  $40,000 @ 3%/yr        │  $60,000 @ illustrated    │
  │  = $100/mo interest     │  cap 10%, floor 0%        │
  │                         │  = $375/mo interest        │
  └─────────────────────────────────────────────────┘
  Total monthly interest = $100 + $375 = $475
```

In [ ]:
from decimal import Decimal

from opie.assumptions.models import IndexAccount
from opie.products.iul import _account_monthly_rate

## Strategy Types

### Fixed Account
The simplest strategy — a guaranteed rate applied monthly.

```
monthly_rate = fixed_rate / 12
```

In [ ]:
fixed = IndexAccount(
    name="Fixed",
    allocation=Decimal("0.40"),
    strategy_type="fixed",
    fixed_rate=Decimal("0.03"),
)

rate = _account_monthly_rate(fixed)
print(f"Fixed account: {fixed.fixed_rate:.1%} annual = {rate} monthly")
print(f"On $40,000: ${Decimal('40000') * rate:,.2f}/month interest")

### Point-to-Point (PTP)

The most common indexed strategy. The crediting rate is derived from the
illustrated index return, subject to a **cap**, **floor**, and **participation rate**.

```
raw_rate     = illustrated_rate × participation
capped_rate  = min(raw_rate, cap)
floored_rate = max(capped_rate, floor)
monthly_rate = floored_rate / 12
```

The floor protects against losses (typically 0%). The cap limits gains.
The participation rate determines what fraction of the index return you get.

In [ ]:
# Scenario: S&P 500 returns 7.5%, cap 10%, floor 0%, 100% participation
ptp = IndexAccount(
    name="S&P 500 PTP",
    allocation=Decimal("0.60"),
    strategy_type="point_to_point",
    illustrated_rate=Decimal("0.075"),
    cap=Decimal("0.10"),
    floor=Decimal("0.00"),
    participation=Decimal("1.00"),
)

rate = _account_monthly_rate(ptp)
print(f"PTP: {ptp.illustrated_rate:.1%} illustrated, {ptp.cap:.0%} cap, {ptp.floor:.0%} floor, {ptp.participation:.0%} participation")
print(f"Monthly rate: {rate}")
print(f"On $60,000: ${Decimal('60000') * rate:,.2f}/month interest")

## Cap/Floor/Participation Sensitivity

The relationship between these parameters is what makes IUL illustrations interesting.
Let's see how different combinations affect the credited rate.

In [ ]:
illustrated = Decimal("0.075")  # 7.5% S&P return

print(f"Illustrated rate: {illustrated:.1%}")
print(f"{'Cap':>6} {'Part':>6} {'Credited':>10} {'Annual Equiv':>14}")
print("-" * 40)

for cap in [Decimal("0.08"), Decimal("0.10"), Decimal("0.12"), Decimal("0.15")]:
    for part in [Decimal("0.80"), Decimal("1.00"), Decimal("1.20")]:
        acct = IndexAccount(
            name="test", allocation=Decimal("1"),
            strategy_type="point_to_point",
            illustrated_rate=illustrated, cap=cap,
            floor=Decimal("0"), participation=part,
        )
        monthly = _account_monthly_rate(acct)
        annual = monthly * 12
        print(f"{cap:>6.0%} {part:>6.0%} {monthly:>10} {annual:>13.4%}")

## Full IUL Illustration

Now let's run a complete illustration and see how the blended crediting
affects account value growth over 20 years.

In [ ]:
import json
from pathlib import Path
from opie import run_illustration
from opie.core.types import IllustrationRequest

payload = json.loads(Path("../examples/iul_request.json").read_text())
request = IllustrationRequest.model_validate(payload)
result = run_illustration(request)

print(f"{'Year':>4}  {'Current AV':>12}  {'Guaranteed AV':>14}  {'Spread':>10}")
print("-" * 46)

for year in [1, 2, 3, 5, 10, 15, 20]:
    c = result.ledgers["current"].rows[year * 12 - 1]
    g = result.ledgers["guaranteed"].rows[year * 12 - 1]
    spread = c.account_value_eop - g.account_value_eop
    print(f"{year:4d}  ${c.account_value_eop:>11,.2f}  ${g.account_value_eop:>13,.2f}  ${spread:>9,.2f}")

## Key Takeaways

1. **Allocation matters**: The split between fixed and indexed accounts determines the risk/return profile
2. **Cap is the ceiling**: Even great index performance gets capped
3. **Floor protects the downside**: The 0% floor means you never lose money from index performance (but charges still apply)
4. **Participation amplifies or dampens**: >100% participation can exceed the illustrated rate (up to the cap)
5. **Guaranteed scenario shows the floor**: With 0% indexed crediting, only the fixed account earns interest

Use `opie compare-strategies` to quickly see how different cap/participation combos affect outcomes:
```bash
opie compare-strategies --in examples/iul_request.json --year 20
```